# Module 5: Multimodal Generative AI Applications
## Topics Covered
- Multimodal AI: Text, Image, Audio, Video Processing
- Speech Recognition & Text-to-Speech (TTS)
- Computer Vision with GPT-4o Vision
- OpenAI Whisper Speech-to-Text
- DALL-E Image Generation
- Image Captioning & Multimodal Retrieval
- Multimodal Chatbots & Cross-Modal Search

**Real-world scenario**: Building a multimodal retail assistant where customers upload product photos, describe items by voice, or ask questions about images.

In [ ]:
%pip install -q openai pillow requests gradio

In [ ]:
import os
import base64
import requests
import json
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your-api-key-here')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)
print('Ready:', OPENAI_API_KEY != 'your-api-key-here')

---
## 1. Multimodal AI Overview

Multimodal AI processes multiple types of data:

| Modality | Input Model | Output Model |
|----------|-------------|-------------|
| Text | GPT-4o | GPT-4o, Claude |
| Image | GPT-4o Vision | DALL-E 3 |
| Audio | Whisper | TTS-1, TTS-1-HD |
| Video | GPT-4o (frames) | Sora (gen) |

---
## 2. Computer Vision with GPT-4o Vision

In [ ]:
def analyze_image_url(image_url: str, question: str) -> str:
    """Analyze an image from URL using GPT-4o vision"""
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image_url',
                        'image_url': {'url': image_url, 'detail': 'high'}
                    },
                    {'type': 'text', 'text': question}
                ]
            }
        ],
        max_tokens=500
    )
    return response.choices[0].message.content

# Test with public image
sample_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg'
questions = [
    'What do you see in this image? Describe in detail.',
    'What are the main colors visible?',
]

for q in questions:
    print(f'Question: {q}')
    answer = analyze_image_url(sample_url, q)
    print(f'Answer: {answer}')
    print('-' * 60)

---
## 3. Image Captioning for Product Catalog

In [ ]:
def generate_product_caption(image_url: str) -> dict:
    """Generate structured product metadata from image using GPT-4o"""
    prompt = '''Analyze this image and return ONLY a JSON object (no markdown) with:
{
  "short_caption": "One sentence description",
  "category": "Product category",
  "tags": ["tag1", "tag2", "tag3"],
  "primary_color": "Main color",
  "seo_keywords": ["keyword1", "keyword2"]
}'''
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image_url', 'image_url': {'url': image_url}},
                {'type': 'text', 'text': prompt}
            ]
        }],
        max_tokens=300
    )
    
    content = response.choices[0].message.content.strip()
    content = content.replace('```json', '').replace('```', '').strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {'raw': content}

product_image = 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg'
caption_data = generate_product_caption(product_image)
print('Generated Product Metadata:')
print(json.dumps(caption_data, indent=2))

---
## 4. DALL-E Image Generation

In [ ]:
from IPython.display import Image, display

def generate_product_image(description: str) -> str:
    """Generate product image with DALL-E 3"""
    prompt = f"""Professional product photography of {description}.
    Clean white background, studio lighting, commercial photography."""
    
    response = client.images.generate(
        model='dall-e-3',
        prompt=prompt,
        size='1024x1024',
        quality='standard',
        n=1,
        style='vivid'
    )
    
    image_url = response.data[0].url
    revised = response.data[0].revised_prompt
    print(f'Generated image for: {description}')
    print(f'DALL-E revised prompt: {revised[:120]}...')
    return image_url

# Generate product images
descriptions = [
    'a sleek wireless bluetooth headphone in midnight black with gold trim',
    'a colorful ceramic coffee mug with geometric patterns in teal and gold'
]

for desc in descriptions:
    print(f'\n{"="*60}')
    url = generate_product_image(desc)
    display(Image(url=url, width=400))

---
## 5. OpenAI Whisper — Speech-to-Text

In [ ]:
def transcribe_audio_api(audio_file_path: str, language: str = 'en') -> dict:
    """
    Transcribe audio using OpenAI Whisper API.
    Supports: mp3, mp4, wav, webm, m4a (max 25MB)
    """
    with open(audio_file_path, 'rb') as audio_file:
        transcript = client.audio.transcriptions.create(
            model='whisper-1',
            file=audio_file,
            language=language,
            response_format='verbose_json',
            timestamp_granularities=['word']
        )
    
    return {
        'text': transcript.text,
        'language': transcript.language,
        'duration_secs': transcript.duration,
        'word_count': len(transcript.text.split())
    }

# Usage:
# result = transcribe_audio_api('customer_inquiry.mp3')
# print(result['text'])

# Local Whisper (free, no API key needed)
def transcribe_local(audio_path: str, model_size: str = 'base') -> str:
    """
    Local Whisper. Install: pip install openai-whisper
    model_size options: tiny(39M), base(74M), small(244M), medium(769M), large(1.5G)
    """
    try:
        import whisper
        model = whisper.load_model(model_size)
        result = model.transcribe(audio_path)
        return result['text']
    except ImportError:
        return 'Install: pip install openai-whisper'

# Whisper model comparison
import pandas as pd
whisper_models = pd.DataFrame({
    'Parameters': ['39M', '74M', '244M', '769M', '1.5G'],
    'Speed vs Large': ['32x', '16x', '6x', '2x', '1x'],
    'WER (lower=better)': ['~15%', '~12%', '~9%', '~6%', '~5%'],
    'Best for': ['Mobile/edge', 'General', 'Balanced', 'Production', 'High accuracy']
}, index=['tiny', 'base', 'small', 'medium', 'large'])

print('Whisper Model Comparison:')
print(whisper_models.to_string())

---
## 6. Text-to-Speech

In [ ]:
def text_to_speech(text: str, voice: str = 'alloy', output_file: str = 'output.mp3') -> str:
    """
    Convert text to speech using OpenAI TTS.
    Voices: alloy (neutral), echo (male), fable (british), onyx (deep), nova (female), shimmer (soft)
    Models: tts-1 (fast, lower quality) | tts-1-hd (slower, higher quality)
    """
    response = client.audio.speech.create(
        model='tts-1',
        voice=voice,
        input=text,
        speed=1.0  # 0.25 to 4.0
    )
    response.stream_to_file(output_file)
    print(f'Saved: {output_file} | Voice: {voice}')
    return output_file

# Generate multiple voice announcements
announcements = [
    ('alloy',   'Welcome to our store. Today only: 30 percent off all electronics.'),
    ('nova',    'Your order has been confirmed and will ship within 2 business days.'),
    ('onyx',    'Thank you for shopping with us. Your satisfaction is our priority.'),
]

for voice, text in announcements:
    text_to_speech(text, voice=voice, output_file=f'announcement_{voice}.mp3')
    print(f'  Text: {text[:60]}...')

---
## 7. Cross-Modal Search: Text Query finds Products by Visual Description

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document

# Product catalog with both visual and text descriptions
# Visual descriptions would come from DALL-E captions or manual curation
products = [
    {'id': 'p001', 'name': 'Wireless Headphones XZ500',
     'visual': 'over-ear matte black headphones with gold accents, cushioned ear pads, folding arms',
     'features': 'ANC, 40hr battery, Bluetooth 5.3', 'price': 199.99},
    {'id': 'p002', 'name': 'Ceramic Pour-Over Coffee Set',
     'visual': 'white ceramic dripper with blue hand-painted flowers, matching mug, wooden stand',
     'features': 'artisan, dishwasher safe, 600ml capacity', 'price': 65.00},
    {'id': 'p003', 'name': 'Bamboo Mechanical Keyboard',
     'visual': 'compact keyboard with natural bamboo case, brown switches, warm wood tones',
     'features': 'Cherry MX Brown switches, USB-C, TKL layout', 'price': 149.99},
    {'id': 'p004', 'name': 'Smart LED Desk Lamp',
     'visual': 'minimalist chrome arm with circular LED ring, USB-C port on circular base',
     'features': 'adjustable color temperature, wireless charging base', 'price': 89.99},
    {'id': 'p005', 'name': 'Full-Grain Leather Messenger Bag',
     'visual': 'rich brown leather bag with brass buckles, multiple pockets, crossbody strap',
     'features': 'genuine leather, fits 15-inch laptop, handcrafted', 'price': 299.00},
]

# Index combined visual + feature descriptions
docs = [
    Document(
        page_content=f"{p['visual']}. Features: {p['features']}",
        metadata={'name': p['name'], 'price': p['price'], 'id': p['id']}
    ) for p in products
]

vectorstore = FAISS.from_documents(docs, OpenAIEmbeddings(model='text-embedding-3-small'))

def cross_modal_search(query: str, k: int = 3):
    """Search products using natural language - finds by visual AND feature similarity"""
    results = vectorstore.similarity_search_with_score(query, k=k)
    print(f'Search: "{query}"')
    for doc, score in results:
        print(f'  [{score:.3f}] {doc.metadata["name"]} - ${doc.metadata["price"]}')
    print()

# Natural language queries that find products by visual characteristics
cross_modal_search('something with warm wood and natural materials for my desk')
cross_modal_search('elegant gift for coffee enthusiast who appreciates handcraft')
cross_modal_search('noise canceling headphones with long battery life')
cross_modal_search('charging my phone while working, minimalist style')

---
## 8. Multimodal Chatbot with Gradio

In [ ]:
import gradio as gr
import base64
import io

def multimodal_retail_bot(message, image, history):
    """
    Multimodal retail assistant that accepts text + images.
    Analyzes product images and answers shopping questions.
    """
    content = []
    
    # Add image if provided
    if image is not None:
        from PIL import Image as PILImage
        pil_img = PILImage.fromarray(image)
        buffer = io.BytesIO()
        pil_img.save(buffer, format='JPEG')
        img_b64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
        content.append({
            'type': 'image_url',
            'image_url': {'url': f'data:image/jpeg;base64,{img_b64}'}
        })
    
    content.append({'type': 'text', 'text': message or 'Describe this product image'})
    
    messages = [{
        'role': 'system',
        'content': 'You are a helpful retail assistant. Analyze product images, describe features, suggest similar items, help customers decide. Be concise.'
    }]
    
    for h_user, h_ai in history[-4:]:
        messages.append({'role': 'user', 'content': h_user})
        messages.append({'role': 'assistant', 'content': h_ai})
    
    messages.append({'role': 'user', 'content': content})
    
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages,
            max_tokens=400
        )
        return response.choices[0].message.content
    except Exception as e:
        return f'Error: {str(e)}'

with gr.Blocks(theme=gr.themes.Soft(), title='Multimodal Retail Assistant') as app:
    gr.Markdown('# Multimodal Retail Assistant')
    gr.Markdown('Upload a product image and ask questions about it!')
    
    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=400)
            with gr.Row():
                msg = gr.Textbox(placeholder='Ask about this product...', scale=4)
                btn = gr.Button('Send', scale=1)
        with gr.Column(scale=1):
            image_in = gr.Image(label='Upload Product Photo')
    
    state = gr.State([])
    
    def respond(message, image, history):
        response = multimodal_retail_bot(message, image, history)
        history.append((message, response))
        return '', gr.update(value=history), history
    
    btn.click(respond, [msg, image_in, state], [msg, chatbot, state])
    msg.submit(respond, [msg, image_in, state], [msg, chatbot, state])

app.launch(share=False)

---
## Summary

| Modality | Technology | API / Library |
|----------|-----------|---------------|
| Vision | GPT-4o Vision | `client.chat.completions` with `image_url` |
| Image Gen | DALL-E 3 | `client.images.generate` |
| Speech-to-Text | Whisper API | `client.audio.transcriptions.create` |
| Local STT | openai-whisper | `whisper.load_model().transcribe()` |
| Text-to-Speech | TTS-1 | `client.audio.speech.create` |
| Cross-modal search | FAISS + Embeddings | Index captions, query with natural language |
| Multimodal UI | Gradio | `gr.Image()` + `gr.ChatInterface` |

### Next: Module 6 — AI Agents with function calling and tool orchestration